# ShopEase AI Customer Support Chatbot — Colab
Runs the chatbot from your ShopEase project: policy RAG (21 support topics) + personalised order help (cancel/return eligibility, ETA, spending, recommendations) + product search, in a Gradio chat window.
**Run the cells in order** and upload `shopease.zip` when asked. Uses your **Google Gemini API key** for natural replies; works without a key too (plain answers from your data).

In [ ]:
!pip install -q gradio requests httpx mongomock-motor motor bcrypt pyjwt "pydantic[email]"

In [ ]:
import os, zipfile
if not os.path.exists("shopease/backend"):
    from google.colab import files
    up = files.upload()                      # choose shopease.zip
    zipfile.ZipFile(next(iter(up))).extractall("shopease")
print(os.listdir("shopease"))

In [ ]:
# Gemini API key: add GEMINI_API_KEY under Colab "Secrets" (key icon, enable notebook access), or paste it when asked.
import os, getpass
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = None
if not key:
    key = getpass.getpass("Gemini API key (press Enter to skip): ").strip()
if key:
    os.environ["GEMINI_API_KEY"] = key
os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash")   # change if your key/quota uses another model
print("Mode:", f"Gemini ({os.environ['GEMINI_MODEL']})" if key else "Plain answers from policies/orders/products (no LLM)")

In [ ]:
# Load the project's chatbot + seed an in-memory database (184 products, users, sample orders, 11 policies)
import sys, os
os.environ["MONGO_URL"] = "mock"
sys.path.insert(0, os.path.abspath("shopease/backend"))
import seed
await seed.seed()
from app.db import db
from app.chatbot.engine import answer, call_gemini
print("Ready. Policy sections indexed for retrieval.")

In [ ]:
# Check that your Gemini key works (skipped if no key)
if os.environ.get("GEMINI_API_KEY"):
    try:
        print("Gemini says:", await call_gemini("Reply briefly.", [{"role": "user", "content": "Say: key works"}]))
    except Exception as e:
        print("Gemini call failed:", e, "\n-> check the key, or try GEMINI_MODEL='gemini-2.0-flash' / 'gemini-2.5-flash-lite'. The chatbot will still answer in plain mode.")

In [ ]:
# Quick test (guest vs logged-in customer)
demo_user = await db.users.find_one({"email": "user@shopease.com"}, {"_id": 0, "password": 0})
orders = await db.orders.find({"user_id": "U0001"}, {"_id": 0}).to_list(10)
print("Your sample order IDs:", [o["order_id"] for o in orders], "\n")
for who, q in [(None, "What is your return policy?"), (None, "Can I cancel after it has shipped?"), (None, "track my order"),
               (demo_user, f"Where is my order {orders[0]['order_id']}?"), (None, "warranty on the Samsung Galaxy S24"), (None, "asdf qwerty")]:
    r = await answer(q, "test", who)
    print(f"[{'user' if who else 'guest'}] {q}\n  -> {r['reply'][:200]!r}\n  sources: {r['sources']}\n")

In [ ]:
import gradio as gr

GUEST, USER = "Guest (not logged in)", "Demo User (user@shopease.com)"

async def respond(message, history, who, request: gr.Request):
    user = await db.users.find_one({"email": "user@shopease.com"}, {"_id": 0, "password": 0}) if who == USER else None
    r = await answer(message, request.session_hash or "anon", user)
    src = f"\n\n_Sources: {', '.join(r['sources'])}_" if r["sources"] else ""
    tips = f"\n\n💡 _You can also ask: {' | '.join(r['options'][:4])}_" if r.get("options") else ""
    return r["reply"] + src + tips

demo = gr.ChatInterface(
    respond,
    additional_inputs=[gr.Dropdown([GUEST, USER], value=USER, label="Chat as (order lookups need a logged-in user)")],
    examples=[["What is your return policy?", USER], ["How long does delivery take and what are the charges?", GUEST],
              ["Where is my order " + orders[0]["order_id"] + "?", USER], ["Which payment methods do you accept?", GUEST],
              ["What is the warranty on the Samsung Galaxy S24?", GUEST], ["Track my order", GUEST],
              ["What did I order?", USER], ["Can I cancel my latest order?", USER], ["Can I return my last delivered order?", USER], ["Recommend products for me", USER],
              ["Recommend laptops under 60000", GUEST], ["How do coupons work?", GUEST], ["I received a damaged item", GUEST], ["Do you have EMI?", GUEST]],
    title="  🛍️ ShopEase Support Assistant",
    description="Ask about shipping, returns, refunds, cancellations, warranty, payments, products or your orders.",
)
demo.launch(share=True, debug=True)